In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

In [ ]:
ds = load_dataset(
    "parquet",
    data_files="https://huggingface.co/datasets/ds-training-nba/nba_shot_data/resolve/main/raw_merged/merged_dataset.parquet"
)
df = ds["train"].to_pandas()

In [ ]:
df.head()

In [ ]:
summary_data = []

for i, col in enumerate(df.columns):
    # Basic Info
    col_type = df[col].dtype
    rows_missing = df[col].isna().sum()
    missing_percent = (rows_missing / len(df.index)) * 100
    
    # Logic for Distribution Column
    distribution = ""
    
    # Check if the column is numerical
    if pd.api.types.is_bool_dtype(df[col]):
        desc = df[col].describe()
        # Formatting describe() output 
        distribution = (f"count: {desc['count']}, most frequent entry: {desc['top']}, amount: {desc['freq']}")
    elif pd.api.types.is_numeric_dtype(df[col]):
        desc = df[col].describe()
        # Formatting describe() output 
        distribution = (f"Mean: {desc['mean']:.2f}, Std: {desc['std']:.2f}, "
                        f"Min: {desc['min']}, 25%: {desc['25%']}, "
                        f"50%: {desc['50%']}, 75%: {desc['75%']}, Max: {desc['max']}")   
    # else categorical
    else:
        unique_values = df[col].unique()
        if len(unique_values) < 10:
            distribution = f"Unique values: {list(unique_values)}"
        else:
            distribution = f"{len(unique_values)} unique values"

    # Append row data
    summary_data.append({
        "#Column": i + 1,
        "Name of the Column": col,
        "Description": "",
        "Variable's type": str(col_type),
        "Rows with missing values": rows_missing,
        "Percentage of missing values": f"{missing_percent:.2f}%",
        "Categorical / Quantitative Distribution": distribution
    })

# Create the summary DataFrame
summary_df = pd.DataFrame(summary_data)

In [ ]:
summary_df

In [ ]:
# Export to Excel
summary_df.to_csv('../data/data_audit_generated.csv', index=False)

# Additional manual checks

In [ ]:
df['PERIOD_x'].value_counts(normalize=True)*100

In [ ]:
df[['GAME_EVENT_ID', 'actionNumber']]
df['GAME_EVENT_ID'].equals(df['actionNumber'].astype('Int64'))

(df['GAME_EVENT_ID'] == df['actionNumber'].astype('Int64')).all()

In [ ]:
(df['SHOT_DISTANCE'] - df['shotDistance']).describe()

In [ ]:
df.loc[~df[['GAME_ID', 'GAME_EVENT_ID']].duplicated(keep=False), 'isFieldGoal'].value_counts()

In [ ]:
df[['isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal']].head(20)

In [ ]:
list(df['shotValue'].unique())
#df[['PERSON1TYPE', 'PLAYER1_NAME', 'PLAYER1_TEAM_CITY', 'shotValue']]
#df['NEUTRALDESCRIPTION'].dropna()

In [ ]:
df.shape